# ▶️ Executed proof — Notebook 03 pipeline on SYNTHETIC data

The real `FlyRank/internship-warehouse` is **gated** and needs *your* Hugging Face token, which this sandbox has no access to. To show the pipeline actually executes, the setup cell builds **local parquet with the same columns** the pipeline expects, then runs the **real, hardened cells 6→8→10→12** from `03_working_with_the_full_release.ipynb`. Numbers are synthetic — the point is the code runs end-to-end and produces the model report.

In [1]:
# OFFLINE DEMO SETUP - stands in for the gated Steps 1-4 (no HF token needed).
# Builds LOCAL parquet with the SAME columns the pipeline expects, then runs the
# real cells 6->8->10->12 below. Numbers are synthetic; the point is: it runs.
import duckdb
from pathlib import Path
D = Path("work/_demo"); D.mkdir(parents=True, exist_ok=True)
g = duckdb.connect()

g.execute(f"""COPY (
  SELECT 'client_' || (c % 5)                              AS client_hash_id,
         (['full','search_only','partial'])[1 + (c % 3)]   AS access_profile,
         (DATE '2025-02-01' + CAST(c % 300 AS INTEGER))   AS gsc_data_start,
         (DATE '2025-04-01' + CAST(c % 200 AS INTEGER))   AS ga4_data_start
  FROM generate_series(0, 4) t(c)
) TO '{D}/dim_clients.parquet' (FORMAT PARQUET)""")

g.execute(f"""COPY (
  SELECT (DATE '2026-06-30' - CAST(d AS INTEGER))          AS report_date,
         'client_' || (c % 5)                              AS client_hash_id,
         'content_' || printf('%04d', c)                   AS content_hash_id,
         CAST(5 + (abs(hash(c*97 + d)) % 60) AS INTEGER)   AS gsc_impressions,
         CAST(abs(hash(c*13 + d)) % 6 AS INTEGER)          AS gsc_clicks,
         3.0 + (abs(hash(c + d*7)) % 400) / 10.0           AS gsc_avg_position
  FROM generate_series(0, 480) t1(c) CROSS JOIN generate_series(0, 89) t2(d)
) TO '{D}/fact_daily.parquet' (FORMAT PARQUET)""")

g.execute(f"""COPY (
  SELECT 'content_' || printf('%04d', c)                   AS content_hash_id,
         CAST(10 + (abs(hash(c*7 + q)) % 500) AS INTEGER)  AS query_impressions_90d,
         (abs(hash(c))   % 30) / 100.0                     AS rare_impressions_share,
         (abs(hash(c*2)) % 20) / 100.0                     AS anonymized_impressions_share,
         CAST(3000 + (abs(hash(c)) % 5000) AS INTEGER)     AS content_total_impressions_90d
  FROM generate_series(0, 480) t1(c)
  CROSS JOIN generate_series(0, 5) t2(q)
) TO '{D}/fact_query.parquet' (FORMAT PARQUET)""")

con = duckdb.connect()
TABLES = {
    'dim_clients':    f"read_parquet('{D}/dim_clients.parquet')",
    'fact_daily':     f"read_parquet('{D}/fact_daily.parquet')",
    'fact_query_90d': f"read_parquet('{D}/fact_query.parquet')",
}
print("Synthetic tables ready. Running the real pipeline cells below...")


Synthetic tables ready. Running the real pipeline cells below...


## 2. Panel / clients *(real cell)*

In [2]:
clients = con.sql(f"""
    SELECT client_hash_id, access_profile, gsc_data_start, ga4_data_start
    FROM {TABLES['dim_clients']}
    ORDER BY gsc_data_start NULLS LAST
""").df()

print('clients with 12+ months of GSC history:',
      (clients['gsc_data_start'] <= clients['gsc_data_start'].dropna().max() - __import__('pandas').Timedelta(days=365)).sum())
clients.head(10)


clients with 12+ months of GSC history: 0


,client_hash_id,access_profile,gsc_data_start,ga4_data_start
0,client_0,full,2025-02-01,2025-04-01
1,client_1,search_only,2025-02-02,2025-04-02
2,client_2,partial,2025-02-03,2025-04-03
3,client_3,full,2025-02-04,2025-04-04
4,client_4,search_only,2025-02-05,2025-04-05


## 3. Build features with SQL *(real cell)*

In [3]:
features = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']}
    ),
    windowed AS (
        SELECT f.client_hash_id, f.content_hash_id,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last30,
               SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_clicks ELSE 0 END)      AS clk_last30,
               AVG(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_avg_position END)       AS pos_last30
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.report_date > b.end_d - INTERVAL 60 DAY
        GROUP BY 1, 2
        HAVING imp_prev30 >= 100
    )
    SELECT * FROM windowed
""").df()

print(f'{len(features):,} content items with enough history')
features.head()


481 content items with enough history


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30
0,client_0,content_0000,1060.0,1019.0,88.0,23.666667
1,client_2,content_0007,952.0,1208.0,78.0,24.713333
2,client_0,content_0010,1001.0,976.0,80.0,20.603333
3,client_4,content_0014,1053.0,1069.0,89.0,24.796667
4,client_4,content_0029,893.0,1073.0,74.0,24.266667


## 4. Query-level signals *(real, hardened cell)*

In [4]:
# Query-mix signals, resolved from the table's ACTUAL columns so a naming
# difference can't crash the run. Guaranteed to emit the 4 columns the model uses.
import re

qcols = con.sql(f"DESCRIBE SELECT * FROM {TABLES['fact_query_90d']}").df()['column_name'].tolist()
q_impr = next((c for c in qcols
               if re.search('impression', c, re.I)
               and 'share' not in c.lower() and 'total' not in c.lower()), None)
has_total = 'content_total_impressions_90d' in qcols
has_rare  = 'rare_impressions_share' in qcols
has_anon  = 'anonymized_impressions_share' in qcols

sel = ["content_hash_id",
       "COUNT(*) AS visible_queries",
       ("ANY_VALUE(rare_impressions_share)" if has_rare else "CAST(NULL AS DOUBLE)") + " AS rare_share",
       ("ANY_VALUE(anonymized_impressions_share)" if has_anon else "CAST(NULL AS DOUBLE)") + " AS anon_share"]
if q_impr and has_total:
    sel.append(f"MAX({q_impr}) / NULLIF(ANY_VALUE(content_total_impressions_90d), 0) AS top_query_share")
else:
    sel.append("CAST(NULL AS DOUBLE) AS top_query_share")

qsignals = con.sql(f"""
    SELECT {', '.join(sel)}
    FROM {TABLES['fact_query_90d']}
    GROUP BY content_hash_id
""").df()

data = features.merge(qsignals, on='content_hash_id', how='left')
print(f'joined: {len(data):,} rows  |  query impressions col: {q_impr}')
data.head()

joined: 481 rows  |  query impressions col: query_impressions_90d


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30,visible_queries,rare_share,anon_share,top_query_share
0,client_0,content_0000,1060.0,1019.0,88.0,23.666667,6,0.00,0.00,0.164000
1,client_2,content_0007,952.0,1208.0,78.0,24.713333,6,0.26,0.16,0.055483
2,client_0,content_0010,1001.0,976.0,80.0,20.603333,6,0.18,0.14,0.125478
3,client_4,content_0014,1053.0,1069.0,89.0,24.796667,6,0.26,0.06,0.055574
4,client_4,content_0029,893.0,1073.0,74.0,24.266667,6,0.28,0.02,0.093434


## 5. A first honest model *(real, hardened cell)*

In [5]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

data['is_declining'] = (data['imp_last30'] < 0.8 * data['imp_prev30']).astype(int)

feature_cols = ['imp_prev30', 'visible_queries', 'rare_share', 'anon_share', 'top_query_share']
model_data = data.dropna(subset=feature_cols)
X, y = model_data[feature_cols], model_data['is_declining']

# stratify only when every class has >=2 samples (else train_test_split raises)
strat = y if (y.nunique() > 1 and y.value_counts().min() >= 2) else None
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=strat)
model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr, y_tr)

print(f'base rate (always predict majority): {max(y_te.mean(), 1 - y_te.mean()):.3f}')
print(classification_report(y_te, model.predict(X_te), digits=3))

base rate (always predict majority): 0.959
              precision    recall  f1-score   support

           0      0.958     0.983     0.970       116
           1      0.000     0.000     0.000         5

    accuracy                          0.942       121
   macro avg      0.479     0.491     0.485       121
weighted avg      0.918     0.942     0.930       121

